# Notebook 08 — End-to-End Baseline Integration

**Objective.** Process a video from start to finish with one call, using the components validated so far, and show that it holds together: events keep their identity through every step, the result can be repeated, and a step that fails or is unsure is handled and reported.

**Background.** Each earlier notebook validated one part. This one tests the joins between them, on three videos with different rules and conditions. It adds no new method, apart from the function that joins the steps.

**Input.**
- `data/traffic_video_modified.mp4` with `configs/scenes/traffic_video_modified.json`: the **simulated signal**, whose graphic is treated as the real signal. It has a moving camera and readable plates.
- `data/raw/rlr_occasions.mp4` with `configs/scenes/rlr_occasions.json`: real red-light footage of one junction.
- `data/video-test.mp4` with `configs/scenes/video-test.json`: the wrong-way rule, on a clip with no violation.

**Output**, in `outputs/08_end_to_end_baseline/<video name>/` (git-ignored):
- `<video name>_output.mp4`: the annotated video, with each event's plate shown after it happens.
- `<video name>_summary.json`: what happened, and how long each step took.
- `violations/`, `plates/`, `events/`: the outputs of each step, including one clip per event.
- `<video name>_decisions.json`, `<video name>_audit.json`, `violations.db`: the simulated review.

**Plate numbers stay out of git.** A git filter removes every image output from the committed copy of this notebook, and printed text shows only a plate's first two characters and its length.

## 0. Setup

In [1]:
import json
import sqlite3
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from traffic_violation_detection.camera import estimate_camera_motion
from traffic_violation_detection.detection import detect, load_detector
from traffic_violation_detection.events import APPROVED, PENDING, REJECTED, SqliteViolationStore, load_records
from traffic_violation_detection.pipeline import run_pipeline
from traffic_violation_detection.plates import load_plate_models, mask_plate
from traffic_violation_detection.red_light import load_red_light_scene, read_frame_signals, score_candidates
from traffic_violation_detection.tracking import create_tracker, update_tracker
from traffic_violation_detection.video import iter_frames, read_frame_at, read_video_metadata

%config InlineBackend.figure_formats = ["jpeg"]

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"No pyproject.toml found in {start} or any parent directory")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUT_DIR = OUTPUTS / "08_end_to_end_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENES = PROJECT_ROOT / "configs" / "scenes"
SIMULATED = (PROJECT_ROOT / "data" / "traffic_video_modified.mp4", SCENES / "traffic_video_modified.json")
REAL = (PROJECT_ROOT / "data" / "raw" / "rlr_occasions.mp4", SCENES / "rlr_occasions.json")
WRONG_WAY = (PROJECT_ROOT / "data" / "video-test.mp4", SCENES / "video-test.json")

detector = load_detector()
plate_models = load_plate_models()
print("plate models run on:", plate_models[0].worker.providers[0])


def show(image, title=None, dpi=100):
    height, width = image.shape[:2]
    figure, axis = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.axis("off")
    if title:
        axis.set_title(title, fontsize=11, loc="left")
    plt.show()


def print_summary(run):
    summary = run["summary"]
    print(f"rule {summary['rule']}, {summary['frames_analysed']} frames, {summary['seconds_of_video']} s of video")
    print("outcomes:  ", summary["outcomes"])
    print("candidates:", summary["candidates"], "| plates:", summary["plates"], "| review:", summary["review"], "| stored:", summary["stored"])
    print("failures:  ", summary["failures"] or "none", "| warnings:", summary["warnings"] or "none")


def event_lines(records):
    for r in records:
        print(f"  {r['event_id']:<52}{r['time']['trigger_time_s']:>7.2f} s  {r['license_plate']['status']:<15}{str(mask_plate(r['license_plate']['text'])):<9}{r['review']['status']:<9}{', '.join(r['needs_attention']) or '-'}")

plate models run on: CUDAExecutionProvider


INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)


## 1. The pipeline

`run_pipeline` in `src/traffic_violation_detection/pipeline.py` joins the step functions:

| Step | Function | Validated in |
|---|---|---|
| Read frames, detect, track, read the signal, apply the rule | `detect_red_light_video` or `detect_wrong_way_video`, chosen by the scene | Notebooks 01–05b |
| Cut one clip per candidate | `write_event_clips`, inside that step | Notebook 05b |
| Read plates from the clips | `read_plates_from_clips` | Notebook 06 |
| Build records and evidence | `generate_review_events` | Notebook 07 |
| Review and store | `apply_review`, `store_decisions` | Notebook 07 |
| Write the annotated video | `write_final_video` | This notebook |

```python
from traffic_violation_detection.pipeline import run_pipeline

result = run_pipeline("video.mp4", "output_folder", "configs/scenes/scene.json")
```

**A person's review stays outside the pipeline.** Without a `review` argument every record stays `pending` and nothing is stored. In this notebook a script stands in for the reviewer, so nothing here is a confirmed finding.

## 2. From start to finish, on the simulated signal

The scripted reviewer approves a clear case and leaves one with an estimated entry `pending`, as a real reviewer might want a closer look.

In [3]:
def review_simulated(record):
    if "entry_estimated" in record["needs_attention"]:
        return None
    return {"decision": APPROVED, "reviewer": "simulated-reviewer", "notes": "Scripted, on a simulated signal."}


simulated = run_pipeline(*SIMULATED[:1], OUTPUT_DIR / "traffic_video_modified", SIMULATED[1], review=review_simulated, model=detector, plate_models=plate_models)
print_summary(simulated)
event_lines(simulated["decided"])
print("\nfiles in the output folder:", sorted(p.name for p in (OUTPUT_DIR / "traffic_video_modified").iterdir()))

rule red_light, 715 frames, 27.15 s of video
outcomes:   {'signal_not_red': 7, 'not_evaluable_signal': 1, 'candidate': 2, 'stopped_at_line': 1}
candidates: 2 | plates: {'read': 2} | review: {'approved': 1, 'pending': 1} | stored: 1
failures:   none | warnings: none
  evt-traffic_video_modified-main-red_light-f00317-t23  12.04 s  read           YB****   approved simulated_signal
  evt-traffic_video_modified-main-red_light-f00365-t37  13.86 s  read           NN***    pending  simulated_signal, entry_estimated

files in the output folder: ['events', 'plates', 'traffic_video_modified_audit.json', 'traffic_video_modified_decisions.json', 'traffic_video_modified_output.mp4', 'traffic_video_modified_summary.json', 'violations', 'violations.db']


In [4]:
output = simulated["output_video"]
frame = read_frame_at(output, simulated["records"][0]["time"]["trigger_frame"] + 20)["image"]
show(cv2.resize(frame, (960, round(frame.shape[0] * 960 / frame.shape[1])), interpolation=cv2.INTER_AREA), "The final video, shortly after the first event. Local copy only.")
store = SqliteViolationStore(simulated["database"])
print("confirmed in the database:", store.count())
for row in store.connection.execute("SELECT event_id, plate_source, reviewer FROM violations"):
    print(" ", dict(row))
store.close()

confirmed in the database: 1
  {'event_id': 'evt-traffic_video_modified-main-red_light-f00317-t23', 'plate_source': 'ocr', 'reviewer': 'simulated-reviewer'}


## 3. Consistent identities

An event must keep one id through every step. `run_pipeline` checks this and stops if it fails. Here it is checked from the saved files, one by one. The red-light event ids also differ from track ids, and each names its video.

In [5]:
def ids_in_each_file(folder, video_id):
    return {
        "violation events": {e["event_id"] for e in json.loads((folder / "violations" / f"{video_id}_events.json").read_text()).get("events", [])},
        "clips": {c["event_id"] for c in json.loads((folder / "violations" / f"{video_id}_violation_clips" / "manifest.json").read_text())["clips"]},
        "plates": {e["event_id"] for e in json.loads((folder / "plates" / "plates.json").read_text())["events"]},
        "records": {r["event_id"] for r in load_records(folder / "events" / f"{video_id}_review_events.json")},
        "decisions": {r["event_id"] for r in json.loads((folder / f"{video_id}_decisions.json").read_text())},
    }


folder = OUTPUT_DIR / "traffic_video_modified"
sets = ids_in_each_file(folder, "traffic_video_modified")
for name, ids in sets.items():
    print(f"  {name:<18}{len(ids)} ids")
assert all(ids == sets["violation events"] for ids in sets.values())
database_ids = {row[0] for row in sqlite3.connect(folder / "violations.db").execute("SELECT event_id FROM violations")}
assert database_ids <= sets["records"]
print("the same ids in every file, and the database holds", len(database_ids), "of them")

  violation events  2 ids
  clips             2 ids
  plates            2 ids
  records           2 ids
  decisions         2 ids
the same ids in every file, and the database holds 1 of them


**Can the result be repeated?** The same video is processed again from nothing. Everything but the timing and the time of the decisions should be identical.

In [6]:
repeat = run_pipeline(*SIMULATED[:1], OUTPUT_DIR / "repeat", SIMULATED[1], review=review_simulated, model=detector, plate_models=plate_models)


def comparable(run):
    # Evidence paths differ with the folder, so they are left out.
    return [{k: v for k, v in r.items() if k not in ("evidence",)} for r in run["records"]]


print("records identical:", comparable(simulated) == comparable(repeat))
print("outcomes identical:", simulated["summary"]["outcomes"] == repeat["summary"]["outcomes"])
assert comparable(simulated) == comparable(repeat)

records identical: True
outcomes identical: True


## 4. The other two videos

**Real footage.** The pipeline should give what the notebooks gave step by step. The scripted reviewer follows the labels of Notebook 05b, which no person has reviewed: a candidate matching a `violation` label is approved, and any other is rejected. The one rejected here matches an `uncertain` label.

In [7]:
labels = json.loads((PROJECT_ROOT / "data" / "annotations" / "rlr_occasions_labels.json").read_text())
all_labels = {**labels["clips"], **labels["development_clips"]}
limits = labels["matching"]["max_frame_difference"], labels["matching"]["max_x_difference"]


def review_by_label(record):
    candidate = {"frame": record["time"]["trigger_frame"], "x": record["measurements"]["x"]}
    outcome = score_candidates([candidate], all_labels[record["track"]["clip"]]["events"], *limits)
    if outcome["true_positives"]:
        return {"decision": APPROVED, "reviewer": "simulated-reviewer", "notes": "Scripted from a label."}
    return {"decision": REJECTED, "reviewer": "simulated-reviewer", "notes": "Scripted: not a labelled violation."}


real = run_pipeline(*REAL[:1], OUTPUT_DIR / "rlr_occasions", REAL[1], review=review_by_label, model=detector, plate_models=plate_models)
print_summary(real)

earlier = json.loads((OUTPUTS / "05b_red_light_violation" / "rule_v2" / "rlr_occasions_events.json").read_text())
earlier_plates = json.loads((OUTPUTS / "06_license_plate_recognition" / "violation_clips" / "plates.json").read_text())
same_events = {e["event_id"] for e in earlier["events"]} == {r["event_id"] for r in real["records"]}
same_plates = {e["event_id"]: e["license_plate"]["status"] for e in earlier_plates["events"]} == {r["event_id"]: r["license_plate"]["status"] for r in real["records"]}
print("\nsame events as Notebook 05b:", same_events, "| same plate results as Notebook 06:", same_plates)
assert same_events and same_plates

rule red_light, 5150 frames, 171.67 s of video
outcomes:   {'not_evaluable_border': 10, 'candidate': 17, 'not_evaluable_ended': 2, 'stopped_at_line': 6, 'not_evaluable_signal': 4, 'signal_not_red': 1}
candidates: 17 | plates: {'unreadable': 14, 'no_plate_found': 3} | review: {'approved': 16, 'rejected': 1} | stored: 16
failures:   none | warnings: none

same events as Notebook 05b: True | same plate results as Notebook 06: True


**The wrong-way rule, on a clip with no violation.** The scene chooses the rule. With nothing to report, every step should still finish, and say so.

In [8]:
wrong_way = run_pipeline(*WRONG_WAY[:1], OUTPUT_DIR / "video-test", WRONG_WAY[1], review=review_simulated, model=detector, plate_models=plate_models)
print_summary(wrong_way)
earlier = json.loads((OUTPUTS / "05_violation_detection" / "video-test.wrong_way.candidates.json").read_text())
print("\nsame track statuses as Notebook 05:", earlier["status_counts"] == wrong_way["summary"]["outcomes"])
assert earlier["status_counts"] == wrong_way["summary"]["outcomes"] and wrong_way["output_video"].is_file() and wrong_way["records"] == []

rule wrong_way, 900 frames, 15.02 s of video
outcomes:   {'not_in_directional_region': 106, 'no_violation': 20, 'not_evaluable': 10}
candidates: 0 | plates: {} | review: {} | stored: 0
failures:   none | warnings: none

same track statuses as Notebook 05: True


## 5. Unsuccessful and uncertain stages

The pipeline must not turn a failure into a result, or lose what it already found. Each case uses one short clip of the real footage (clip F, 6 s, with one violation) or a small made-up file.

In [9]:
TRIALS = OUTPUT_DIR / "trials"
TRIALS.mkdir(exist_ok=True)
scene = json.loads(REAL[1].read_text())
one_clip = {**scene, "clips": {"F": scene["clips"]["F"]}}
one_clip_path = TRIALS / "one_clip.json"
one_clip_path.write_text(json.dumps(one_clip))


class BrokenPlates:
    def predict(self, image):
        raise RuntimeError("the plate worker stopped")

    def run_one(self, *arguments, **keywords):
        raise RuntimeError("the plate worker stopped")


def report(name, run):
    summary = run["summary"]
    plates = dict(Counter(r["license_plate"]["status"] for r in run["records"]))
    print(f"  {name:<40} candidates {summary['candidates']}, outcomes {summary['outcomes']}, plates {plates or '-'}, failures {len(summary['failures'])}")


print("the same clip, working normally and with a stage failing or unsure:")
normal = run_pipeline(REAL[0], TRIALS / "normal", one_clip_path, model=detector, plate_models=plate_models)
report("all stages working", normal)
assert normal["summary"]["candidates"] == 1

broken = run_pipeline(REAL[0], TRIALS / "plates_broken", one_clip_path, model=detector, plate_models=(BrokenPlates(), BrokenPlates()))
report("plate reading fails", broken)
print("     the failure is reported:", broken["summary"]["failures"][0])
print("     the record says:", broken["records"][0]["needs_attention"])
assert broken["summary"]["candidates"] == 1 and broken["records"][0]["license_plate"]["status"] == "not_attempted"

no_signal = json.loads(json.dumps(one_clip))
no_signal["clips"]["F"]["signals"].update(through=[[300, 200, 340, 240]], left=[[300, 200, 340, 240]])  # A dark area with no signal.
no_signal_path = TRIALS / "no_signal.json"
no_signal_path.write_text(json.dumps(no_signal))
unsure = run_pipeline(REAL[0], TRIALS / "no_signal", no_signal_path, model=detector, plate_models=plate_models)
report("the signal cannot be read", unsure)
assert unsure["summary"]["candidates"] == 0 and unsure["summary"]["outcomes"].get("not_evaluable_signal", 0) >= 1

the same clip, working normally and with a stage failing or unsure:


  all stages working                       candidates 1, outcomes {'candidate': 1, 'stopped_at_line': 1, 'not_evaluable_border': 1}, plates {'unreadable': 1}, failures 0


  plate reading fails                      candidates 1, outcomes {'candidate': 1, 'stopped_at_line': 1, 'not_evaluable_border': 1}, plates {'not_attempted': 1}, failures 1
     the failure is reported: Plate reading failed: the plate worker stopped
     the record says: ['plate_not_attempted']


  the signal cannot be read                candidates 0, outcomes {'not_evaluable_signal': 2, 'not_evaluable_border': 1}, plates -, failures 0


In [10]:
def attempt(name, action):
    try:
        action()
        print(f"  {name:<40} NOT REFUSED")
    except (ValueError, FileNotFoundError, RuntimeError, OSError) as error:
        print(f"  {name:<40} refused: {type(error).__name__}: {str(error)[:70]}")


noise_file = TRIALS / "not_a_video.mp4"
noise_file.write_bytes(np.random.default_rng(0).integers(0, 256, 4096, dtype=np.uint8).tobytes())
print("inputs that cannot be processed, refused before any slow step:")
attempt("a missing video", lambda: run_pipeline(TRIALS / "missing.mp4", TRIALS / "x", REAL[1], model=detector, plate_models=plate_models))
attempt("a file that is not a video", lambda: run_pipeline(noise_file, TRIALS / "x", REAL[1], model=detector, plate_models=plate_models))
attempt("a scene of another size", lambda: run_pipeline(WRONG_WAY[0], TRIALS / "x", REAL[1], model=detector, plate_models=plate_models))
attempt("a missing scene", lambda: run_pipeline(REAL[0], TRIALS / "x", TRIALS / "missing.json", model=detector, plate_models=plate_models))

inputs that cannot be processed, refused before any slow step:
  a missing video                          refused: FileNotFoundError: Video file not found (or not a file): /workspace/traffic-violation-det
  a file that is not a video               refused: RuntimeError: OpenCV could not open the video: /workspace/traffic-violation-detectio
  a scene of another size                  refused: ValueError: Scene is for 1280x720, video is 1920x1080
  a missing scene                          refused: FileNotFoundError: [Errno 2] No such file or directory: '/workspace/traffic-violation-det


[mov,mp4,m4a,3gp,3g2,mj2 @ 0x64d7e3e88b00] moov atom not found


## 6. Performance

All timings are on one RTX 4090. **By step**, for each video:

In [11]:
runs = {"simulated": simulated, "real footage": real, "wrong way": wrong_way}
steps = list(simulated["summary"]["steps_seconds"])
print(f"{'step':<36}" + "".join(f"{name:>15}" for name in runs))
for step in steps:
    print(f"{step:<36}" + "".join(f"{run['summary']['steps_seconds'].get(step, 0):>14.1f}s" for run in runs.values()))
print(f"{'total':<36}" + "".join(f"{run['summary']['total_seconds']:>14.1f}s" for run in runs.values()))
print(f"{'frames':<36}" + "".join(f"{run['summary']['frames_analysed']:>15}" for run in runs.values()))
print(f"{'seconds of video':<36}" + "".join(f"{run['summary']['seconds_of_video']:>14.1f}s" for run in runs.values()))
print(f"{'frames per second':<36}" + "".join(f"{run['summary']['frames_analysed'] / run['summary']['total_seconds']:>15.1f}" for run in runs.values()))
print(f"{'speed against the video':<36}" + "".join(f"{run['summary']['times_real_time']:>14.2f}x" for run in runs.values()))

step                                      simulated   real footage      wrong way
load the detector                              0.0s           0.0s           0.0s
detect, track and apply the rule             125.1s         171.6s         119.0s
read plates                                   41.1s          38.0s           0.0s
build records and evidence                     7.3s          18.0s           0.0s
review and store                               0.0s           0.0s           0.0s
write the final video                         12.1s          26.1s           1.8s
total                                        185.6s         253.8s         120.8s
frames                                          715           5150            900
seconds of video                              27.1s         171.7s          15.0s
frames per second                               3.9           20.3            7.4
speed against the video                       0.15x          0.68x          0.12x


**Inside the first step**, per frame, on 300 frames of the simulated video at 1920×1012:

In [12]:
clip = load_red_light_scene(SIMULATED[1], 1920, 1012)["clips"]["main"]
camera = json.loads(SIMULATED[1].read_text())["camera"]
tracker = create_tracker()
spent = Counter()
detections_by_frame = []
frames = iter_frames(SIMULATED[0], 0, 300)
while True:
    t0 = time.perf_counter()
    record = next(frames, None)
    t1 = time.perf_counter()
    if record is None:
        break
    detections = detect(detector, record["image"])
    t2 = time.perf_counter()
    update_tracker(tracker, detections, record["image"])
    t3 = time.perf_counter()
    read_frame_signals(record["image"], clip)
    t4 = time.perf_counter()
    spent.update(decode=t1 - t0, detect=t2 - t1, track=t3 - t2, signal=t4 - t3)
    detections_by_frame.append(detections)
start = time.perf_counter()
estimate_camera_motion(SIMULATED[0], detections_by_frame, 150, camera["excluded_areas"])
spent["camera motion"] = time.perf_counter() - start
total = sum(spent.values())
for name, seconds in spent.most_common():
    print(f"  {name:<15}{1000 * seconds / 300:>7.1f} ms per frame  {seconds / total:>5.0%}")
print(f"  {'sum':<15}{1000 * total / 300:>7.1f} ms per frame, {300 / total:.1f} frames per second")

  camera motion     67.6 ms per frame    73%
  detect            18.9 ms per frame    21%
  track              3.4 ms per frame     4%
  decode             1.8 ms per frame     2%
  signal             0.3 ms per frame     0%
  sum               92.1 ms per frame, 10.9 frames per second


## Results

The table comes from the outputs above, and figures from separate checks say so. One function took each video from start to finish, and the rule followed from the scene configuration.

| | Simulated signal | Real footage | Wrong way |
|---|---|---|---|
| Frames | 715 | 5,150 | 900 |
| Crossings or tracks judged | 11 | 40 | 136 |
| Candidates | 2 | 17 | 0 |
| Plates read | 2 | 0 | not applicable |
| Scripted decisions | 1 approved, 1 left `pending` | 16 approved, 1 rejected | none |
| Stored | 1 | 16 | 0 |
| Time | 186 s (110 s when repeated) | 254 s | 121 s |
| Speed against the video | 0.15× (0.25× when repeated) | 0.68× | 0.12× |

- **Same results as the notebooks.** The real footage gave the same 17 events as Notebook 05b and the same plate results as Notebook 06, and the wrong-way run gave the same 136 track statuses as Notebook 05.
- **Consistent identities.** Each event kept one id through the violation step, clips, plates, records and decisions, and only the approved one reached the database.
- **Repeatable.** The simulated video processed twice from nothing gave identical records.
- **Failures and uncertainty.** When plate reading failed, the violation was kept, the failure was reported, and the record says `plate_not_attempted`. A signal that cannot be read gave no candidate and `not_evaluable_signal`. A clip with no violation finished every step, and four kinds of bad input were refused before any slow step.
- **A check by eye.** In Notebook 06 every vehicle in both red phases of the simulated video was checked by eye, and all verdicts on red were right. A blue car could not be judged, because a taxi hid it as it crossed.

**Where the time goes.**

- **It is not real time on this hardware.** The fastest run took about 1.5 s per second of video, and the slowest about 8 s. The batch use this project targets can accept that, but a live stream could not.
- **The largest part measured is camera motion,** at 73% of the 92 ms per frame measured in section 6, or 68 ms. Detection takes 19 ms and tracking 3 ms. Fixed cameras skip this step. These parts explain about 66 s of the first step's 83 s to 125 s on this video. The rest, which includes writing the annotated video and the clips, was not profiled.
- **An avoidable slowdown was found.** Importing Ultralytics leaves OpenCV on one thread, which made camera motion about 4 times slower than it needed to be. It now raises the thread count for that step. These figures come from separate runs: OpenCV reported 34 threads before the import and 1 after, the step took about 350 ms per frame before the fix and 81 ms after, and different thread counts gave identical results on all 715 frames.
- **A wrong trial, corrected.** The first trial of an unreadable signal still flagged a car, because its region lay on a real signal head beside the road. The rule was right and the trial was wrong, so the region was moved to a dark area.

## False positives and false negatives

This notebook measures integration, not accuracy, so it adds no new scores. Those of Notebook 05b stand. The simulated run agrees with what can be seen by eye, but that is one short clip.

## Limitations

- **Nobody has reviewed anything.** Decisions are scripted, and so are the labels behind them, so the 17 "stored" records only test the contract.
- **The chain from a violation to a plate has only worked on a simulated signal.** On real footage it stops at the plate, as none could be read.
- **Each camera needs a scene configuration by hand,** and only two kinds exist here. The pipeline cannot process a video it has no scene for.
- **A wrong scene is not detected.** The rule reads whatever is at the declared position. A real signal in the wrong place reads as a valid signal, and even empty, dark areas gave a colour in up to 10 of 178 frames from passing light. The size of the video is checked, but not that the scene fits it.
- **One rule per run.** A scene selects one rule, so a video is not yet checked for several violations at once.
- **Timings are from single runs on one RTX 4090,** with models already loaded, so they exclude loading weights. They are not stable: the simulated video took 186 s the first time and 110 s when repeated, with plate reading going from 41 s to 10 s and the first step from 125 s to 83 s. The first run was the first to use the models, but the cause was not established.
- **Plate reading times cannot be compared between columns.** The first simulated run took 41 s for 2 events, and the real footage 38 s for 17, so how this step grows with events is not yet known.
- **The final video can disagree with itself.** Its top banner comes from the violation step, so it says `pending review` even when the caption below says `approved`.

## Conclusion

**Completion criterion** — *a video can be processed end-to-end using the validated baseline components, with consistent event identities and correct handling of unsuccessful or uncertain stages*:

- **End to end: met,** on three videos with two rules.
- **Consistent identities: met,** in every file and in a repeated run.
- **Unsuccessful and uncertain stages: met** in the cases tried. A failed plate step, an unreadable signal, a clip with no violation and four kinds of bad input were all handled. A video that ends early was not tried: the frame reader warns about it and `run_pipeline` collects the violation step's warnings in its summary, but no pipeline test covers this path.

**Readiness.** `run_pipeline` is in `src/traffic_violation_detection/pipeline.py` with tests, and is the entry point for an application.

**Next steps.**

- **Notebook 09 — evaluation** can draw these results together and rank what to improve in Phase 2.
- **A person's review** of the labels, and **footage with real violations and readable plates,** remain the two things that would change these conclusions most.